### Research Question 3: False-Positive and False-Negative Subgroup Investigation

**Are false positives/negatives disproportionately associated with particular patient or image characteristics, especially AP/PA view, age, sex, and image acquisition?**

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [20]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

In [2]:
# Import data from the base notebook
import import_ipynb
from IPython.utils.capture import capture_output

In [3]:
# Use "capture" do suppress plotting images
with capture_output() as captured:
    %run xraymodel.ipynb

In [12]:
FP_THRESHOLD = 0.7
MIN_GROUP_SIZE = 20  # suppress unstable rates from tiny subgroups

def show_fp_fn_characteristics(min_group_size = MIN_GROUP_SIZE, threshold=0.7):
    # ---- 1. Align metadata with the images in the current dataset ----
    meta = XRAY_DATASET.csv.copy().reset_index(drop=True)
    n_images = len(XRAY_DATASET)
    if len(meta) != n_images:
        raise ValueError(
            f"Metadata has {len(meta)} rows but dataset has {n_images} images. "
            "Check SubsetDataset metadata alignment before continuing."
        )
    if len(XRAY_LABELS_LIST) != n_images or len(MODEL_SCORE_PREDICTIONS_CONVERTED) != n_images:
        raise ValueError("Images, labels and predictions are not aligned in length.")

    # Normalize common metadata fields without assuming every source has them.
    def _first_existing(columns):
        for c in columns:
            if c in meta.columns:
                return c
        return None

    view_col = _first_existing(["AP/PA", "ViewPosition", "View Position", "view"])
    sex_col = _first_existing(["Sex", "Patient Gender", "sex", "Gender"])
    age_col = _first_existing(["Age", "Patient Age", "age"])

    if view_col:
        meta["View"] = meta[view_col].astype(str).str.strip().str.upper()
        meta["View"] = meta["View"].replace({"NAN": "Unknown", "": "Unknown"})
        meta.loc[~meta["View"].isin(["AP", "PA"]), "View"] = "Other/Unknown"
    else:
        meta["View"] = "Unavailable"

    if sex_col:
        meta["Sex"] = meta[sex_col].astype(str).str.strip().str.upper().replace({
            "M": "Male", "MALE": "Male", "F": "Female", "FEMALE": "Female",
            "NAN": "Unknown", "": "Unknown"
        })
        meta.loc[~meta["Sex"].isin(["Male", "Female"]), "Sex"] = "Unknown"
    else:
        meta["Sex"] = "Unavailable"

    if age_col:
        meta["Age_years"] = pd.to_numeric(meta[age_col], errors="coerce")
        # Keep plausible adult/pediatric ages; do not silently treat sentinel values as real ages.
        meta.loc[(meta["Age_years"] < 0) | (meta["Age_years"] > 120), "Age_years"] = np.nan
        meta["Age_group"] = pd.cut(
            meta["Age_years"], bins=[0, 40, 50, 60, 70, 80, 120],
            labels=["≤40", "41–50", "51–60", "61–70", "71–80", ">80"],
            include_lowest=True
        ).astype("object").fillna("Unknown")
    else:
        meta["Age_years"] = np.nan
        meta["Age_group"] = "Unavailable"

    # Candidate acquisition / image metadata fields present in this dataset.
    acquisition_candidates = [
        "Frontal/Lateral", "ViewPosition", "AP/PA", "PatientOrientation",
        "ImageType", "PhotometricInterpretation", "Manufacturer",
        "Institution", "Portable", "AcquisitionDeviceProcessingDescription"
    ]
    acquisition_columns = [c for c in acquisition_candidates if c in meta.columns]
    print("Metadata columns available:", list(meta.columns))
    print("Acquisition-related columns found:", acquisition_columns or "None beyond view metadata")
    print("View counts:\n", meta["View"].value_counts(dropna=False))
    print("Sex counts:\n", meta["Sex"].value_counts(dropna=False))

    # ---- 2. Build long-form outcome table; exclude uncertain/missing labels ----
    # IMPORTANT: CheXpert-style labels may contain -1 (uncertain). They are excluded
    # from binary FP/FN calculations instead of being silently treated as negatives.
    rows = []
    for j, pathology in enumerate(DATASET_PATHOLOGIES):
        truth = np.asarray(XRAY_LABELS_LIST[:, j], dtype=float)
        score = np.asarray(MODEL_SCORE_PREDICTIONS_CONVERTED[:, j], dtype=float)
        valid = np.isfinite(truth) & np.isfinite(score) & np.isin(truth, [0, 1])
        pred = (score >= threshold).astype(int)
        outcome = np.where((truth == 0) & (pred == 1), "False Positive",
                np.where((truth == 1) & (pred == 0), "False Negative",
                np.where((truth == 1) & (pred == 1), "True Positive", "True Negative")))
        tmp = meta.copy()
        tmp["Pathology"] = pathology
        tmp["y_true"] = truth
        tmp["score"] = score
        tmp["y_pred"] = pred
        tmp["Outcome"] = outcome
        tmp["Valid_label"] = valid
        rows.append(tmp.loc[valid])

    experiment_df = pd.concat(rows, ignore_index=True)
    print(f"Valid binary image-pathology observations: {len(experiment_df):,}")
    total_possible = n_images * len(DATASET_PATHOLOGIES)
    print(f"Excluded uncertain/missing observations: {total_possible - len(experiment_df):,}")

    # Wilson 95% confidence interval for a binomial rate.
    def wilson_interval(successes, total, z=1.96):
        if total == 0:
            return np.nan, np.nan
        p = successes / total
        den = 1 + z*z/total
        center = (p + z*z/(2*total)) / den
        half = z * np.sqrt((p*(1-p)/total) + (z*z/(4*total*total))) / den
        return max(0, center-half), min(1, center+half)


    def subgroup_metrics(df, group_col, min_n=MIN_GROUP_SIZE):
        """Calculate FPR and FNR with separate denominators and 95% CIs."""
        records = []

        for (pathology, group), g in df.groupby(
            ["Pathology", group_col], observed=True, dropna=False
        ):
            negatives = g[g["y_true"] == 0]
            positives = g[g["y_true"] == 1]

            n_neg = len(negatives)
            n_pos = len(positives)

            fp = int((negatives["y_pred"] == 1).sum())
            fn = int((positives["y_pred"] == 0).sum())

            fp_lo, fp_hi = wilson_interval(fp, n_neg)
            fn_lo, fn_hi = wilson_interval(fn, n_pos)

            records.append({
                "Pathology": pathology,
                "Group": str(group),
                "N_images": len(g),

                "N_actual_negative": n_neg,
                "FP": fp,
                "FP_rate": fp / n_neg if n_neg else np.nan,
                "FP_CI_low": fp_lo,
                "FP_CI_high": fp_hi,

                "N_actual_positive": n_pos,
                "FN": fn,
                "FN_rate": fn / n_pos if n_pos else np.nan,
                "FN_CI_low": fn_lo,
                "FN_CI_high": fn_hi,

                "Enough_negative": n_neg >= min_n,
                "Enough_positive": n_pos >= min_n
            })

        return pd.DataFrame(records)
    # ---- PLOT 1–3: FP rate by view, sex, and age group across pathologies ----
    def plot_error_rates_by_subgroup(group_col, min_n=MIN_GROUP_SIZE):

        stats = subgroup_metrics(experiment_df, group_col, min_n=min_n)

        for error_type in ["FP", "FN"]:

            rate_col = f"{error_type}_rate"
            ci_low = f"{error_type}_CI_low"
            ci_high = f"{error_type}_CI_high"
            count_col = error_type
            denominator = (
                "N_actual_negative" if error_type == "FP"
                else "N_actual_positive"
            )
            enough_col = (
                "Enough_negative" if error_type == "FP"
                else "Enough_positive"
            )

            plot_data = stats[
                stats[enough_col] & stats[rate_col].notna()
            ].copy()

            if plot_data.empty:
                print(
                    f"No groups for {group_col} have at least "
                    f"{min_n} eligible cases for {error_type}."
                )
                continue

            fig = px.bar(
                plot_data,
                x="Pathology",
                y=rate_col,
                color="Group",
                barmode="group",

                error_y=plot_data[ci_high] - plot_data[rate_col],
                error_y_minus=plot_data[rate_col] - plot_data[ci_low],

                hover_data=[
                    "N_images", denominator, count_col,
                    ci_low, ci_high
                ],

                title=(
                    f"{'False-positive' if error_type == 'FP' else 'False-negative'} "
                    f"rate by {group_col} "
                    f"(threshold={threshold}; minimum n={min_n})"
                ),

                labels={
                    rate_col: (
                        "False-positive rate (among actual negatives)"
                        if error_type == "FP"
                        else "False-negative rate (among actual positives)"
                    ),
                    "Pathology": "Finding"
                }
            )

            fig.update_yaxes(tickformat=".0%", range=[0, 1])
            fig.update_layout(
                template="plotly_white",
                height=600,
                width=1250
            )

            fig.show()

        return stats

    view_stats = plot_error_rates_by_subgroup("View")
    sex_stats = plot_error_rates_by_subgroup("Sex")
    age_stats = plot_error_rates_by_subgroup("Age_group")

    # Also plot other acquisition metadata when it exists and has manageable cardinality.
    # This commonly includes frontal/lateral; AP/PA is already covered above.
    acquisition_stats = {}
    for _col in acquisition_columns:
        if _col in [view_col, "AP/PA", "ViewPosition"]:
            continue
        _n_levels = meta[_col].astype(str).nunique(dropna=False)
        if 2 <= _n_levels <= 12:
            meta[_col + "_group"] = meta[_col].astype(str).replace({"nan": "Unknown", "": "Unknown"})
            # The long table already carries original metadata columns; normalize on that table.
            experiment_df[_col + "_group"] = experiment_df[_col].astype(str).replace({"nan": "Unknown", "": "Unknown"})
            acquisition_stats[_col] = plot_error_rates_by_subgroup(_col + "_group")

    # ---- PLOT 4: AP/PA × sex heatmap for a selected pathology ----
    def plot_view_sex_heatmap(pathology):

        d = experiment_df[
            (experiment_df["Pathology"] == pathology) &
            (experiment_df["View"].isin(["AP", "PA"])) &
            (experiment_df["Sex"].isin(["Male", "Female"]))
        ]

        fig = make_subplots(
            rows=1, cols=2,
            subplot_titles=["False-positive rate", "False-negative rate"]
        )

        for col, error_type, truth_value in [
            (1, "FP", 0),
            (2, "FN", 1)
        ]:

            subset = d[d["y_true"] == truth_value]

            agg = subset.groupby(
                ["View", "Sex"], observed=True
            ).agg(
                Errors=("y_pred", lambda s: int(
                    (s == (1 if error_type == "FP" else 0)).sum()
                )),
                N=("y_pred", "size")
            ).reset_index()

            agg["Rate"] = agg["Errors"] / agg["N"]

            agg["Label"] = agg.apply(
                lambda r: f"{r.Rate:.1%}<br>{error_type}={r.Errors}/{r.N}",
                axis=1
            )

            matrix = agg.pivot(
                index="View", columns="Sex", values="Rate"
            ).reindex(index=["AP", "PA"], columns=["Female", "Male"])

            labels = agg.pivot(
                index="View", columns="Sex", values="Label"
            ).reindex(index=["AP", "PA"], columns=["Female", "Male"])

            fig.add_trace(
                go.Heatmap(
                    z=matrix.values,
                    x=matrix.columns,
                    y=matrix.index,
                    zmin=0,
                    zmax=1,
                    colorscale="Reds" if error_type == "FP" else "Blues",
                    colorbar=dict(
                        title="Error rate",
                        x=0.46 if col == 1 else 1.0
                    ),
                    text=labels.values,
                    texttemplate="%{text}",
                    hovertemplate=(
                        "View: %{y}<br>Sex: %{x}"
                        "<br>Error rate: %{z:.1%}<extra></extra>"
                    ),
                    showscale=True
                ),
                row=1, col=col
            )

        fig.update_layout(
            title=f"{pathology}: FP and FN rates by view × sex",
            template="plotly_white",
            height=500,
            width=1100
        )

        fig.show()
        return fig


    # Example: plot_view_sex_heatmap(DATASET_PATHOLOGIES[0])

    # ---- PLOT 5: UMAP showing whether FP points concentrate by subgroup ----
    def plot_error_embedding(
        pathology,
        error_type="FP",
        subgroup="View",
        projection="UMAP",
        max_points=12000
    ):

        if error_type not in ["FP", "FN"]:
            raise ValueError("error_type must be FP or FN")

        if subgroup not in ["View", "Sex", "Age_group"]:
            raise ValueError("Invalid subgroup")

        label_idx = DATASET_LABEL_INDICES_MAP[pathology]

        truth = np.asarray(XRAY_LABELS_LIST[:, label_idx], dtype=float)
        scores = np.asarray(
            MODEL_SCORE_PREDICTIONS_CONVERTED[:, label_idx],
            dtype=float
        )

        valid = (
            np.isfinite(truth) &
            np.isfinite(scores) &
            np.isin(truth, [0, 1])
        )

        if error_type == "FP":
            error_mask = valid & (truth == 0) & (scores >= threshold)
        else:
            error_mask = valid & (truth == 1) & (scores < threshold)

        if projection == "S-UMAP":
            emb = downprojector.calculate_projection(
                "feature", "S-UMAP",
                target_labels=XRAY_LABELS_LIST,
                label_name=pathology
            )
        else:
            emb = downprojector.calculate_projection(
                "feature", projection
            )

        d = meta.copy()
        d["x"] = emb[:, 0]
        d["y"] = emb[:, 1]
        d["score"] = scores

        keep = valid.copy()

        if keep.sum() > max_points:
            rng = np.random.default_rng(RANDOM_STATE)
            keep_idx = rng.choice(
                np.flatnonzero(keep),
                max_points,
                replace=False
            )
            keep = np.zeros(len(d), dtype=bool)
            keep[keep_idx] = True

        fig = go.Figure()

        # Background: all valid cases
        fig.add_trace(go.Scattergl(
            x=d.loc[keep, "x"],
            y=d.loc[keep, "y"],
            mode="markers",
            name="All valid cases",
            marker=dict(color="lightgrey", size=4, opacity=0.25),
            hoverinfo="skip"
        ))

        # Highlight errors, separated by subgroup
        for group in sorted(
            d.loc[error_mask, subgroup].astype(str).unique()
        ):

            m = (
                error_mask &
                d[subgroup].astype(str).eq(group).to_numpy() &
                keep
            )

            fig.add_trace(go.Scattergl(
                x=d.loc[m, "x"],
                y=d.loc[m, "y"],
                mode="markers",
                name=f"{error_type} · {group}",
                marker=dict(size=7, opacity=0.85),
                customdata=np.column_stack([
                    d.loc[m, "score"],
                    d.loc[m, "Age_years"]
                ]),
                hovertemplate=(
                    f"{subgroup}: {group}"
                    "<br>Score: %{customdata[0]:.3f}"
                    "<br>Age: %{customdata[1]}"
                    "<extra></extra>"
                )
            ))

        fig.update_layout(
            title=(
                f"{pathology}: {error_type} cases in {projection} "
                f"coloured by {subgroup}"
            ),
            xaxis_title=f"{projection} 1",
            yaxis_title=f"{projection} 2",
            template="plotly_white",
            width=1000,
            height=750
        )

        fig.show()
        return fig

    # Examples:
    # plot_fp_embedding("Cardiomegaly", subgroup="View")
    # plot_fp_embedding("Cardiomegaly", subgroup="Sex")
    # plot_fp_embedding("Cardiomegaly", subgroup="Age_group")

    # ---- PLOT 6: Score distributions among actual negatives by subgroup ----
    def plot_score_distributions(pathology, subgroup="View"):

        for truth_value, error_type, title in [
            (0, "FP", "Actual negatives (FP analysis)"),
            (1, "FN", "Actual positives (FN analysis)")
        ]:

            d = experiment_df[
                (experiment_df["Pathology"] == pathology) &
                (experiment_df["y_true"] == truth_value)
            ].copy()

            fig = px.violin(
                d,
                x=subgroup,
                y="score",
                color=subgroup,
                box=True,
                points="outliers",
                title=f"{pathology}: {title} by {subgroup}",
                labels={
                    "score": "Model prediction score",
                    subgroup: subgroup
                }
            )

            fig.add_hline(
                y=threshold,
                line_dash="dash",
                line_color="black",
                annotation_text=f"Threshold = {threshold}"
            )

            fig.update_yaxes(range=[0, 1])

            fig.update_layout(
                template="plotly_white",
                showlegend=False,
                height=550,
                width=900
            )

            fig.show()
    return experiment_df



In [13]:
show_fp_fn_characteristics()

Metadata columns available: ['PatientID', 'Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices', 'view', 'patientid', 'age_years', 'sex_male', 'sex_female', 'View', 'Age_years', 'Age_group']
Acquisition-related columns found: ['Frontal/Lateral', 'AP/PA']
View counts:
 View
PA    5663
AP    4337
Name: count, dtype: int64
Sex counts:
 Sex
Male      5800
Female    4200
Name: count, dtype: int64
Valid binary image-pathology observations: 110,000
Excluded uncertain/missing observations: 0


,PatientID,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,...,sex_female,View,Age_years,Age_group,Pathology,y_true,score,y_pred,Outcome,Valid_label
0,patient00001,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68.0,Frontal,AP,1.0,0.0,0.0,0.0,...,True,AP,68.0,61–70,Atelectasis,0.0,0.639400,0,True Negative,True
1,patient00002,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87.0,Frontal,AP,NaN,NaN,-1.0,1.0,...,True,AP,87.0,>80,Atelectasis,0.0,0.812196,1,False Positive,True
2,patient00003,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41.0,Frontal,AP,NaN,NaN,NaN,NaN,...,False,AP,41.0,41–50,Atelectasis,0.0,0.403673,0,True Negative,True
3,patient00004,CheXpert-v1.0-small/train/patient00004/study1/...,Female,20.0,Frontal,PA,1.0,0.0,0.0,0.0,...,True,PA,20.0,≤40,Atelectasis,0.0,0.225516,0,True Negative,True
4,patient00005,CheXpert-v1.0-small/train/patient00005/study1/...,Male,33.0,Frontal,PA,1.0,0.0,0.0,0.0,...,False,PA,33.0,≤40,Atelectasis,0.0,0.323448,0,True Negative,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109995,patient09998,CheXpert-v1.0-small/train/patient09998/study1/...,Female,88.0,Frontal,AP,NaN,0.0,NaN,NaN,...,True,AP,88.0,>80,Pneumothorax,0.0,0.554095,0,True Negative,True
109996,patient09999,CheXpert-v1.0-small/train/patient09999/study1/...,Female,50.0,Frontal,AP,1.0,0.0,0.0,0.0,...,True,AP,50.0,41–50,Pneumothorax,0.0,0.208101,0,True Negative,True
109997,patient10000,CheXpert-v1.0-small/train/patient10000/study1/...,Male,64.0,Frontal,AP,1.0,0.0,0.0,0.0,...,False,AP,64.0,61–70,Pneumothorax,0.0,0.337597,0,True Negative,True
109998,patient10001,CheXpert-v1.0-small/train/patient10001/study2/...,Female,30.0,Frontal,AP,1.0,0.0,0.0,0.0,...,True,AP,30.0,≤40,Pneumothorax,0.0,0.121643,0,True Negative,True


### Answering the Questions

**False-positive rate: AP vs PA**: The diagram shows that the false-positive rate for AP is overall higher than for PA. The only exception is Lung lesion where PA is higher.

**False-negative rate: AP VS PA**: Here the diagram shows that PA has a higher false-negative reate overall, but less dramatic as the false positive rate of AP vs PA.

**False-positive rate: male vs female**: The diagrams does not show significant differences between male and female. 

**False-positive rate: age**: The diagrams shows that in particular the age group **>80** has a high false-positive rate.

### Rerun with threshold of 0.5

In [14]:
show_fp_fn_characteristics(0.5)

Metadata columns available: ['PatientID', 'Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices', 'view', 'patientid', 'age_years', 'sex_male', 'sex_female', 'View', 'Age_years', 'Age_group']
Acquisition-related columns found: ['Frontal/Lateral', 'AP/PA']
View counts:
 View
PA    5663
AP    4337
Name: count, dtype: int64
Sex counts:
 Sex
Male      5800
Female    4200
Name: count, dtype: int64
Valid binary image-pathology observations: 110,000
Excluded uncertain/missing observations: 0


,PatientID,Path,Sex,Age,Frontal/Lateral,AP/PA,No Finding,Enlarged Cardiomediastinum,Cardiomegaly,Lung Opacity,...,sex_female,View,Age_years,Age_group,Pathology,y_true,score,y_pred,Outcome,Valid_label
0,patient00001,CheXpert-v1.0-small/train/patient00001/study1/...,Female,68.0,Frontal,AP,1.0,0.0,0.0,0.0,...,True,AP,68.0,61–70,Atelectasis,0.0,0.639400,0,True Negative,True
1,patient00002,CheXpert-v1.0-small/train/patient00002/study2/...,Female,87.0,Frontal,AP,NaN,NaN,-1.0,1.0,...,True,AP,87.0,>80,Atelectasis,0.0,0.812196,1,False Positive,True
2,patient00003,CheXpert-v1.0-small/train/patient00003/study1/...,Male,41.0,Frontal,AP,NaN,NaN,NaN,NaN,...,False,AP,41.0,41–50,Atelectasis,0.0,0.403673,0,True Negative,True
3,patient00004,CheXpert-v1.0-small/train/patient00004/study1/...,Female,20.0,Frontal,PA,1.0,0.0,0.0,0.0,...,True,PA,20.0,≤40,Atelectasis,0.0,0.225516,0,True Negative,True
4,patient00005,CheXpert-v1.0-small/train/patient00005/study1/...,Male,33.0,Frontal,PA,1.0,0.0,0.0,0.0,...,False,PA,33.0,≤40,Atelectasis,0.0,0.323448,0,True Negative,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
109995,patient09998,CheXpert-v1.0-small/train/patient09998/study1/...,Female,88.0,Frontal,AP,NaN,0.0,NaN,NaN,...,True,AP,88.0,>80,Pneumothorax,0.0,0.554095,0,True Negative,True
109996,patient09999,CheXpert-v1.0-small/train/patient09999/study1/...,Female,50.0,Frontal,AP,1.0,0.0,0.0,0.0,...,True,AP,50.0,41–50,Pneumothorax,0.0,0.208101,0,True Negative,True
109997,patient10000,CheXpert-v1.0-small/train/patient10000/study1/...,Male,64.0,Frontal,AP,1.0,0.0,0.0,0.0,...,False,AP,64.0,61–70,Pneumothorax,0.0,0.337597,0,True Negative,True
109998,patient10001,CheXpert-v1.0-small/train/patient10001/study2/...,Female,30.0,Frontal,AP,1.0,0.0,0.0,0.0,...,True,AP,30.0,≤40,Pneumothorax,0.0,0.121643,0,True Negative,True


### Answering the Questions

The observations stay roughly the same as with a threshold of 0.7.

### Age × View analysis

In [10]:
def age_view_error_rates(long_df, min_n=20):

    records = []

    for (pathology, age, view), g in long_df.groupby(
        ["Pathology", "Age_group", "View"],
        observed=True,
        dropna=False
    ):

        negatives = g[g["y_true"] == 0]
        positives = g[g["y_true"] == 1]

        n_neg = len(negatives)
        n_pos = len(positives)

        fp = (negatives["y_pred"] == 1).sum()
        fn = (positives["y_pred"] == 0).sum()

        records.append({
            "Pathology": pathology,
            "Age_group": age,
            "View": view,

            "N_negative": n_neg,
            "False_positives": fp,
            "FP_rate": fp / n_neg if n_neg else np.nan,

            "N_positive": n_pos,
            "False_negatives": fn,
            "FN_rate": fn / n_pos if n_pos else np.nan,

            "Enough_FP_data": n_neg >= min_n,
            "Enough_FN_data": n_pos >= min_n
        })

    return pd.DataFrame(records)


In [16]:
def plot_age_view_heatmap(stats, pathology, error_type="FN"):

    if error_type == "FN":
        rate_col = "FN_rate"
        enough_col = "Enough_FN_data"
        title = "False-negative rate"
    else:
        rate_col = "FP_rate"
        enough_col = "Enough_FP_data"
        title = "False-positive rate"

    data = stats[
        (stats["Pathology"] == pathology) &
        (stats[enough_col])
    ].copy()

    pivot = data.pivot(
        index="Age_group",
        columns="View",
        values=rate_col
    )

    fig = px.imshow(
        pivot,
        text_auto=".1%",
        color_continuous_scale="Reds",
        zmin=0,
        zmax=1,
        labels={
            "x": "Image view",
            "y": "Age group",
            "color": "Error rate"
        },
        title=f"{title}: {pathology} by Age and View"
    )

    fig.show()

In [15]:
long_df = show_fp_fn_characteristics()
age_view_stats = age_view_error_rates(long_df)

display(age_view_stats.head(20))

Metadata columns available: ['PatientID', 'Path', 'Sex', 'Age', 'Frontal/Lateral', 'AP/PA', 'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices', 'view', 'patientid', 'age_years', 'sex_male', 'sex_female', 'View', 'Age_years', 'Age_group']
Acquisition-related columns found: ['Frontal/Lateral', 'AP/PA']
View counts:
 View
PA    5663
AP    4337
Name: count, dtype: int64
Sex counts:
 Sex
Male      5800
Female    4200
Name: count, dtype: int64
Valid binary image-pathology observations: 110,000
Excluded uncertain/missing observations: 0


,Pathology,Age_group,View,N_negative,False_positives,FP_rate,N_positive,False_negatives,FN_rate,Enough_FP_data,Enough_FN_data
0,Atelectasis,41–50,AP,437,157,0.359268,99,39,0.393939,True,True
1,Atelectasis,41–50,PA,691,126,0.182344,117,72,0.615385,True,True
2,Atelectasis,51–60,AP,602,287,0.476744,164,41,0.250000,True,True
3,Atelectasis,51–60,PA,1026,233,0.227096,204,97,0.475490,True,True
4,Atelectasis,61–70,AP,606,365,0.602310,154,36,0.233766,True,True
5,Atelectasis,61–70,PA,951,279,0.293375,230,103,0.447826,True,True
6,Atelectasis,71–80,AP,572,394,0.688811,140,33,0.235714,True,True
7,Atelectasis,71–80,PA,636,263,0.413522,183,66,0.360656,True,True
8,Atelectasis,>80,AP,697,531,0.761836,143,19,0.132867,True,True
9,Atelectasis,>80,PA,414,239,0.577295,94,26,0.276596,True,True


In [18]:
for pathology in long_df["Pathology"].unique():
    plot_age_view_heatmap(age_view_stats, pathology, "FP")

### Linear Regression

In [21]:
# Select one finding
pathology = "Atelectasis"

# Keep only this pathology and actual negative cases
reg_df = long_df[
    (long_df["Pathology"] == pathology) &
    (long_df["y_true"] == 0)
].copy()

# Define the outcome: 1 = false positive, 0 = true negative
reg_df["error"] = (reg_df["y_pred"] == 1).astype(int)

# Keep AP and PA images and known demographic groups
reg_df = reg_df[
    reg_df["View"].isin(["AP", "PA"]) &
    reg_df["Age_group"].isin(
        ["≤40", "41–50", "51–60", "61–70", "71–80", ">80"]
    ) &
    reg_df["Sex"].isin(["Female", "Male"])
].copy()

# Set reference categories
age_order = ["≤40", "41–50", "51–60", "61–70", "71–80", ">80"]

reg_df["Age_group"] = pd.Categorical(
    reg_df["Age_group"],
    categories=age_order,
    ordered=False
)

reg_df["View"] = pd.Categorical(
    reg_df["View"],
    categories=["PA", "AP"]
)

reg_df["Sex"] = pd.Categorical(
    reg_df["Sex"],
    categories=["Female", "Male"]
)

print("Number of observations:", len(reg_df))
print("Number of false positives:", reg_df["error"].sum())
print("False-positive rate:", round(reg_df["error"].mean() * 100, 2), "%")

display(
    reg_df.groupby(["Age_group", "View"], observed=False)
    .agg(
        N=("error", "size"),
        False_positives=("error", "sum"),
        FP_rate=("error", "mean")
    )
    .reset_index()
)

Number of observations: 8289
Number of false positives: 3098
False-positive rate: 37.37 %


,Age_group,View,N,False_positives,FP_rate
0,≤40,PA,1029,78,0.075802
1,≤40,AP,628,146,0.232484
2,41–50,PA,691,126,0.182344
3,41–50,AP,437,157,0.359268
4,51–60,PA,1026,233,0.227096
5,51–60,AP,602,287,0.476744
6,61–70,PA,951,279,0.293375
7,61–70,AP,606,365,0.602310
8,71–80,PA,636,263,0.413522
9,71–80,AP,572,394,0.688811


In [22]:
# Model 1: Age + View + Sex
model_without_interaction = smf.glm(
    formula="error ~ C(Age_group) + C(View) + C(Sex)",
    data=reg_df,
    family=sm.families.Binomial()
).fit()

# Model 2: Age + View + Age × View + Sex
model_with_interaction = smf.glm(
    formula="error ~ C(Age_group) * C(View) + C(Sex)",
    data=reg_df,
    family=sm.families.Binomial()
).fit()

print(model_with_interaction.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:                  error   No. Observations:                 8289
Model:                            GLM   Df Residuals:                     8276
Model Family:                Binomial   Df Model:                           12
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -4628.3
Date:                Mon, 09 Nov 2026   Deviance:                       9256.5
Time:                        16:56:40   Pearson chi2:                 8.29e+03
No. Iterations:                     5   Pseudo R-squ. (CS):             0.1854
Covariance Type:            nonrobust                                         
                                          coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------------------
In

In [23]:
# Likelihood-ratio test
lr_stat = (
    model_without_interaction.deviance -
    model_with_interaction.deviance
)

df_diff = (
    model_without_interaction.df_resid -
    model_with_interaction.df_resid
)

interaction_p = chi2.sf(lr_stat, df_diff)

print("Likelihood-ratio test for Age × View interaction")
print("-----------------------------------------------")
print("LR statistic:", round(lr_stat, 3))
print("Degrees of freedom:", int(df_diff))
print("P-value:", interaction_p)

if interaction_p < 0.05:
    print("\nThe Age × View interaction is statistically significant.")
    print("The AP–PA difference in false-positive odds varies across age groups.")
else:
    print("\nThe Age × View interaction is not statistically significant.")
    print("There is insufficient evidence that the AP–PA difference varies across age groups.")

Likelihood-ratio test for Age × View interaction
-----------------------------------------------
LR statistic: 9.912
Degrees of freedom: 5
P-value: 0.0777659076286826

The Age × View interaction is not statistically significant.
There is insufficient evidence that the AP–PA difference varies across age groups.


In [24]:
# Extract coefficients and confidence intervals
conf = model_with_interaction.conf_int()

odds_ratios = pd.DataFrame({
    "Coefficient": model_with_interaction.params,
    "Odds_ratio": np.exp(model_with_interaction.params),
    "CI_lower": np.exp(conf[0]),
    "CI_upper": np.exp(conf[1]),
    "p_value": model_with_interaction.pvalues
})

display(odds_ratios.round(4))

,Coefficient,Odds_ratio,CI_lower,CI_upper,p_value
Intercept,-2.5575,0.0775,0.0610,0.0984,0.0000
C(Age_group)[T.41–50],1.0002,2.7187,2.0121,3.6735,0.0000
C(Age_group)[T.51–60],1.2745,3.5768,2.7217,4.7004,0.0000
C(Age_group)[T.61–70],1.6190,5.0481,3.8543,6.6115,0.0000
C(Age_group)[T.71–80],2.1542,8.6210,6.5176,11.4032,0.0000
C(Age_group)[T.>80],2.8171,16.7285,12.3644,22.6328,0.0000
C(View)[T.AP],1.3085,3.7007,2.7526,4.9753,0.0000
C(Sex)[T.Male],0.0928,1.0972,0.9917,1.2139,0.0721
C(Age_group)[T.41–50]:C(View)[T.AP],-0.3814,0.6829,0.4560,1.0227,0.0642
C(Age_group)[T.51–60]:C(View)[T.AP],-0.1713,0.8425,0.5838,1.2159,0.3599
